In [17]:
import numpy as np
import pandas as pd

# ---------------- SETTINGS ----------------
INPUT_PATH = r"C:\Users\Downloads\FIFA Wc2026_raw_dataset.csv"
OUTPUT_PATH = r"C:\Users\Downloads\wc2026_clean.csv"

ID_COL = "Match_ID"
TEAM_COLS = ["Team_A", "Team_B"]
TARGET = "Goal_Difference"
FEATURES = ["Attempts_Diff", "OnTarget_Diff", "OffTarget_Diff", "InsideBox_Diff",
            "Headed_Diff", "xG_Diff", "Corners_Diff", "Possession_Diff"]
# ------------------------------------------

# 1. Load
df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()
print("Shape:", df.shape)

missing_cols = [c for c in [ID_COL, TARGET] + TEAM_COLS + FEATURES if c not in df.columns]
if missing_cols:
    raise SystemExit(f"These columns are not in the file: {missing_cols}\nColumns found: {list(df.columns)}")

# 2. Clean text and numbers
for c in TEAM_COLS:
    df[c] = df[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

for c in [TARGET] + FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")   # bad text becomes NaN

# 3. Checks
print("\n--- CHECKS ---")
print("Rows:", len(df), "(expected 104)")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Match_IDs:", df[ID_COL].duplicated().sum())
print("Team playing itself:", (df[TEAM_COLS[0]] == df[TEAM_COLS[1]]).sum())
print("Distinct teams:", pd.concat([df[TEAM_COLS[0]], df[TEAM_COLS[1]]]).nunique(), "(max 48)")

print("\nMissing values per column:")
miss = df.isna().sum()
print(miss[miss > 0].to_string() if miss.any() else "None")

print("\nGoal_Difference non-whole numbers:", (df[TARGET].dropna() % 1 != 0).sum())
print("Possession_Diff outside -100..100:", (df["Possession_Diff"].abs() > 100).sum())
print("Note: negative values are normal here because every variable is a difference (Team_A minus Team_B).")

print("\nSummary of variables:")
print(df[[TARGET] + FEATURES].describe().round(2).T[["min", "mean", "max", "std"]])

# 4. Outliers (flag only, nothing is removed)
print("\nPossible outliers (|z| > 3):")
found = False
for c in [TARGET] + FEATURES:
    s = df[c]
    if s.std() > 0:
        z = (s - s.mean()) / s.std()
        rows = df.loc[z.abs() > 3, ID_COL].tolist()
        if rows:
            found = True
            print(f"  {c}: Match_ID {rows}")
if not found:
    print("  None")

# 5. Correlation between predictors
corr = df[FEATURES].corr()
print("\nHighly correlated predictors (|r| > 0.8):")
hit = False
for i, a in enumerate(FEATURES):
    for b in FEATURES[i + 1:]:
        if abs(corr.loc[a, b]) > 0.8:
            hit = True
            print(f"  {a} vs {b}: r = {corr.loc[a, b]:.2f}")
if not hit:
    print("  None")

# 6. Drop rows with missing values and save
before = len(df)
df_clean = df.dropna(subset=[TARGET] + FEATURES).drop_duplicates()
print("\nRows dropped (missing or duplicate):", before - len(df_clean))
df_clean.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH, "| Final rows:", len(df_clean))

Shape: (104, 12)

--- CHECKS ---
Rows: 104 (expected 104)
Duplicate rows: 0
Duplicate Match_IDs: 0
Team playing itself: 0
Distinct teams: 48 (max 48)

Missing values per column:
None

Goal_Difference non-whole numbers: 0
Possession_Diff outside -100..100: 0
Note: negative values are normal here because every variable is a difference (Team_A minus Team_B).

Summary of variables:
                    min   mean     max    std
Goal_Difference   -4.00   0.48    6.00   2.02
Attempts_Diff   -104.00  11.29  123.00  47.86
OnTarget_Diff    -51.00   4.80   57.00  21.12
OffTarget_Diff   -41.00   4.39   54.00  18.74
InsideBox_Diff   -68.00   8.06   76.00  31.39
Headed_Diff      -22.00   1.74   24.00   8.84
xG_Diff          -14.79   1.67   15.96   6.46
Corners_Diff     -41.00   4.46   47.00  18.58
Possession_Diff  -27.00   2.38   31.00  12.09

Possible outliers (|z| > 3):
  None

Highly correlated predictors (|r| > 0.8):
  Attempts_Diff vs OnTarget_Diff: r = 0.97
  Attempts_Diff vs OffTarget_Diff: r

In [14]:
"""
FIFA World Cup 2026 - Dataset 1 (v2): clean dataset builder.

Inputs (same folder):
    wc2026_clean.csv                        -> official match list (Match_ID, teams, goals or Goal_Difference)
    fifa_wc2026_team_statistics*.xlsx       -> team statistics

Outputs (same folder):
    wc2026_dataset1_v2_clean.csv
    wc2026_dataset1_v2_validation_report.txt

Target:     Goal_Difference = Team_A goals - Team_B goals
Predictors: Team_A statistic minus Team_B statistic (8 variables, see STAT_MAP).
"""
from pathlib import Path
import unicodedata

import pandas as pd

BASE = Path(r"C:\Users\Pervez Ali\Downloads")
MATCH_CSV = BASE / "wc2026_clean.csv"
OUT_CSV = BASE / "wc2026_dataset1_v2_clean.csv"
OUT_REPORT = BASE / "wc2026_dataset1_v2_validation_report.txt"

TARGET = "Goal_Difference"
STAT_MAP = {
    "Attempts At Goal": "Attempts_Diff",
    "Attempts On Target": "OnTarget_Diff",
    "Off-Target Attempts": "OffTarget_Diff",
    "Attempts Inside the Penalty Area": "InsideBox_Diff",
    "Headed Attempts at Goal": "Headed_Diff",
    "xG": "xG_Diff",
    "Corners": "Corners_Diff",
    "Possession Control (%)": "Possession_Diff",
}
PREDICTORS = list(STAT_MAP.values())

# possible goal-column pairs in the CSV, used only if Goal_Difference is not already there
GOAL_PAIRS = [("Team_A_Goals", "Team_B_Goals"), ("Goals_A", "Goals_B"),
              ("Score_A", "Score_B"), ("Team_A_Score", "Team_B_Score"),
              ("Home_Goals", "Away_Goals"), ("Team_A_goals", "Team_B_goals")]

report = []


def log(msg=""):
    print(msg)
    report.append(str(msg))


def norm(name):
    """Accent- and case-insensitive team key ('Türkiye' == 'Turkiye')."""
    s = unicodedata.normalize("NFKD", str(name).strip())
    return "".join(c for c in s if not unicodedata.combining(c)).lower()


def find_stats_file():
    files = sorted(BASE.glob("fifa_wc2026_team_statistics*.xlsx"))
    if not files:
        raise FileNotFoundError(f"Team statistics .xlsx not found in {BASE}")
    return files[0]


def load_matches():
    df = pd.read_csv(MATCH_CSV)
    df.columns = [c.strip() for c in df.columns]

    if TARGET not in df.columns:
        for a, b in GOAL_PAIRS:
            if a in df.columns and b in df.columns:
                df[TARGET] = pd.to_numeric(df[a]) - pd.to_numeric(df[b])
                break
        else:
            raise KeyError(f"No Goal_Difference or goal columns in {MATCH_CSV.name}. "
                           f"Columns found: {list(df.columns)}")

    df = df[["Match_ID", "Team_A", "Team_B", TARGET]].copy()
    for c in ("Team_A", "Team_B"):
        df[c] = df[c].astype(str).str.strip()

    before = len(df)
    df = df.drop_duplicates(subset="Match_ID").reset_index(drop=True)   # unique matches only
    log(f"Matches loaded from {MATCH_CSV.name}: {before} | after removing duplicate Match_ID: {len(df)}")
    return df


def load_team_stats(path):
    raw = pd.read_excel(path, sheet_name=0, header=None)
    header_row = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq("Team").any(), axis=1)][0]
    st = raw.iloc[header_row + 1:].copy()
    st.columns = [str(c).strip() for c in raw.iloc[header_row]]
    st = st.dropna(subset=["Team"])[["Team"] + list(STAT_MAP)].copy()
    st[list(STAT_MAP)] = st[list(STAT_MAP)].apply(pd.to_numeric, errors="coerce")
    st["key"] = st["Team"].map(norm)
    log(f"Team statistics file: {path.name} | teams: {len(st)} | duplicates: {int(st.key.duplicated().sum())}")
    return st


def build_dataset(matches, stats):
    keyA, keyB = matches.Team_A.map(norm), matches.Team_B.map(norm)
    missing = sorted((set(keyA) | set(keyB)) - set(stats.key))
    log(f"Teams without statistics: {missing if missing else 'none'}")

    table = stats.set_index("key")[list(STAT_MAP)]
    diff = (table.reindex(keyA).reset_index(drop=True)
            - table.reindex(keyB).reset_index(drop=True)).rename(columns=STAT_MAP)
    out = pd.concat([matches.reset_index(drop=True), diff], axis=1)
    out[PREDICTORS] = out[PREDICTORS].round(4)
    out[TARGET] = out[TARGET].astype(int)
    return out.sort_values("Match_ID").reset_index(drop=True)


def validate(df, stats):
    log("\n[1] Missing values per column")
    log(df.isna().sum().to_string())
    log(f"Total missing: {int(df.isna().sum().sum())}")

    log("\n[2] Error checks")
    s = stats
    checks = {
        "Duplicate Match_ID": int(df.Match_ID.duplicated().sum()),
        "Match_ID not exactly 1..104": int(sorted(df.Match_ID) != list(range(1, 105))),
        "Team_A == Team_B": int((df.Team_A.map(norm) == df.Team_B.map(norm)).sum()),
        "Negative raw statistics": int((s[list(STAT_MAP)] < 0).sum().sum()),
        "Possession outside 0-100": int((~s["Possession Control (%)"].between(0, 100)).sum()),
        "On-target > attempts": int((s["Attempts On Target"] > s["Attempts At Goal"]).sum()),
        "Inside-box > attempts": int((s["Attempts Inside the Penalty Area"] > s["Attempts At Goal"]).sum()),
        "Goal_Difference not whole number": int((df[TARGET] % 1 != 0).sum()),
        "|Goal_Difference| > 10": int((df[TARGET].abs() > 10).sum()),
    }
    for name, n in checks.items():
        log(f"{name}: {n}")
    assert not any(checks.values()), "Validation errors found - see report"


def main():
    matches = load_matches()
    stats = load_team_stats(find_stats_file())
    df = build_dataset(matches, stats)
    validate(df, stats)

    assert df.shape == (104, 4 + len(PREDICTORS)) and df.isna().sum().sum() == 0
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    log(f"\nSaved {OUT_CSV.name}: {df.shape[0]} rows x {df.shape[1]} cols")
    OUT_REPORT.write_text("\n".join(report), encoding="utf-8")


if __name__ == "__main__":
    main()

KeyError: "No Goal_Difference or goal columns in wc2026_clean.csv. Columns found: ['Match_ID', 'Team_A', 'Team_B']"